In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import MinMaxScaler


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /Users/isabel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_DIR = 'fromGoogleDrive'

In [4]:
# preprocessing (same as compcor)
def preprocess(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
    text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
    words = word_tokenize(text) # tokenize
    return words

In [5]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):

    # Process texts (lowercase, remove punctuation and numbers).
    list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
    # Flatten words.
    all_words = [word for doc in list_of_texts for word in doc]
    total_words = len(all_words)
    freq = Counter(all_words)
    # Filter stopwords.
    filtered_words = [w for w in all_words if w not in stop_words]
    filtered_word_total = len(filtered_words)
    stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
    # Average word length.
    avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
    # Get document lengths.
    doc_lengths = np.array([len(t) for t in list_of_texts])
    # Vocab.
    vocab = set(all_words)
    vocab_len = len(vocab)
    # Type-Token Ratio.
    ttr = vocab_len / total_words if total_words > 0 else 0
    # Rare words (hapax legomena).
    hapax = sum(1 for _, c in freq.items() if c == 1)

    # Rare words (dis legomena).
    dis = sum(1 for _, c in freq.items() if c == 2)

    # Top-N Coverage (Frequency Concentration)
    top_10_count = sum(c for _, c in freq.most_common(10))
    coverage = top_10_count / total_words if total_words > 0 else 0
    # Append everything to a row. 
    rows.append({
    "Dataset": file.replace('.csv', ''),
    "Total Documents": len(list_of_texts),
    "Total Words": total_words,
    "Total Words (without stopwords)": filtered_word_total,
    "Total Stop Words": total_words - filtered_word_total,
    "Stopword Ratio": stopword_ratio,
    "Average Word Length": avg_word_len,
    "Vocab Length": vocab_len,
    "Type-Token Ratio": ttr,
    "Hapax Legomena": hapax,
    "Dis Legomena": dis,
    "Top 10 Words Coverage": coverage,
    "Mean Document Length": doc_lengths.mean(),
    "Median Document Length": np.median(doc_lengths),
    "Standard Deviation Document Length": doc_lengths.std(),
    "Min Document Length": doc_lengths.min(),
    "Max Document Length": doc_lengths.max(),
    "25th Document Length Percentile": np.percentile(doc_lengths, 25),
    "75th Document Length Percentile": np.percentile(doc_lengths, 75),
    })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=['Total Documents', 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,0.379056,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,0.343715,4.360843,0.035360,0.267681,46.278934,17.0
4,dementiaAudio,0.438171,3.480664,0.026194,0.386441,116.338798,105.0
5,huffPostNews,0.343072,4.199670,0.023536,0.238591,25.163032,23.0
6,medicalAbstracts,0.286589,5.109807,0.021201,0.263213,205.660064,200.0
7,simSUM,0.101771,4.106722,0.012611,0.381137,104.821400,103.0
8,syntheticCareHomeNurseNotes,0.318930,4.937623,0.027334,0.277734,26.532596,24.0
9,yahoo,0.387502,3.922505,0.029172,0.255604,47.845493,42.0


In [6]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):
    if 'dementia' not in file:

        # Process texts (lowercase, remove punctuation and numbers).
        list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
        # Flatten words.
        all_words = [word for doc in list_of_texts for word in doc]
        total_words = len(all_words)
        freq = Counter(all_words)
        # Filter stopwords.
        filtered_words = [w for w in all_words if w not in stop_words]
        filtered_word_total = len(filtered_words)
        stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
        # Average word length.
        avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
        # Get document lengths.
        doc_lengths = np.array([len(t) for t in list_of_texts])
        # Vocab.
        vocab = set(all_words)
        vocab_len = len(vocab)
        # Type-Token Ratio.
        ttr = vocab_len / total_words if total_words > 0 else 0
        # Rare words (hapax legomena).
        hapax = sum(1 for _, c in freq.items() if c == 1)

        # Rare words (dis legomena).
        dis = sum(1 for _, c in freq.items() if c == 2)

        # Top-N Coverage (Frequency Concentration)
        top_10_count = sum(c for _, c in freq.most_common(10))
        coverage = top_10_count / total_words if total_words > 0 else 0
        # Append everything to a row. 
        rows.append({
        "Dataset": file.replace('.csv', ''),
        "Total Documents": len(list_of_texts),
        "Total Words": total_words,
        "Total Words (without stopwords)": filtered_word_total,
        "Total Stop Words": total_words - filtered_word_total,
        "Stopword Ratio": stopword_ratio,
        "Average Word Length": avg_word_len,
        "Vocab Length": vocab_len,
        "Type-Token Ratio": ttr,
        "Hapax Legomena": hapax,
        "Dis Legomena": dis,
        "Top 10 Words Coverage": coverage,
        "Mean Document Length": doc_lengths.mean(),
        "Median Document Length": np.median(doc_lengths),
        "Standard Deviation Document Length": doc_lengths.std(),
        "Min Document Length": doc_lengths.min(),
        "Max Document Length": doc_lengths.max(),
        "25th Document Length Percentile": np.percentile(doc_lengths, 25),
        "75th Document Length Percentile": np.percentile(doc_lengths, 75),
        })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=['Total Documents', 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,0.379056,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,0.343715,4.360843,0.035360,0.267681,46.278934,17.0
4,huffPostNews,0.343072,4.199670,0.023536,0.238591,25.163032,23.0
5,medicalAbstracts,0.286589,5.109807,0.021201,0.263213,205.660064,200.0
6,simSUM,0.101771,4.106722,0.012611,0.381137,104.821400,103.0
7,syntheticCareHomeNurseNotes,0.318930,4.937623,0.027334,0.277734,26.532596,24.0
8,yahoo,0.387502,3.922505,0.029172,0.255604,47.845493,42.0


In [7]:
def make_output_metric_ranking(df):
    normalized_df = df.copy()
    normalized_df = normalized_df
    metric_cols = [
        "Accuracy",
        "Weighted Accuracy",
        "Time",
        "Monotonicity",
        "Separability",
        "Linearity"
    ]

    for col in metric_cols:
        min_val = normalized_df[col].min()
        max_val = normalized_df[col].max()
        normalized_df[col] = (normalized_df[col] - min_val) / (max_val - min_val)

    winners = {f"{metric}": [] for metric in metric_cols}
    for dataset in list(set(normalized_df['dataset1'])):
            # print(f"---------------- {dataset} ----------------")
            temp_df = normalized_df[(normalized_df['dataset1'] == dataset) | (normalized_df['dataset2'] == dataset)]
            temp_df['Overall'] = temp_df[metric_cols].mean(axis=1)
            temp_df.groupby('metric')[metric_cols].mean()
            grouped_df = temp_df.groupby('metric')[metric_cols].mean()
            grouped_df['Algorithm'] = grouped_df.index
            for metric in metric_cols:
                format_df = pl.from_pandas(grouped_df.sort_values(by=metric, ascending=False))
                winners[metric].append(format_df['Algorithm'][0])

    normalized_df['Overall'] = normalized_df[metric_cols].mean(axis=1)
    sorted_mean_df = normalized_df.groupby('metric')[metric_cols + ['Overall']].mean()
    sorted_mean_df['Algorithm'] = sorted_mean_df.index
    polars_output = pl.from_pandas(sorted_mean_df.sort_values(by='Overall', ascending=False))
    print(polars_output)



    all_winners = []
    for metric, values in winners.items():
        print(metric, Counter(values))
        all_winners.extend(values)

    Counter(all_winners)
    return polars_output

In [8]:
all_temp_dfs = []
for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/ksc'):
    if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/ksc/{combination}'):
        combination_splits = combination.split('_')
        dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
        if 'dementia' not in dataset1 and 'dementia' not in dataset2:
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/ksc/{combination}/{combination}_ksc_metrics_measures.csv')
            temp_df = temp_df.groupby('metric').mean().reset_index()
            temp_df['dataset1'] = dataset1
            temp_df['dataset2'] = dataset2
            temp_df['repetitions'] = repetitions
            all_temp_dfs.append(temp_df)

df_ksc = pd.concat(all_temp_dfs)
ksc = make_output_metric_ranking(df_ksc)


shape: (217, 8)
┌──────────┬─────────────┬──────────┬─────────────┬────────────┬───────────┬──────────┬────────────┐
│ Accuracy ┆ Weighted    ┆ Time     ┆ Monotonicit ┆ Separabili ┆ Linearity ┆ Overall  ┆ Algorithm  │
│ ---      ┆ Accuracy    ┆ ---      ┆ y           ┆ ty         ┆ ---       ┆ ---      ┆ ---        │
│ f64      ┆ ---         ┆ f64      ┆ ---         ┆ ---        ┆ f64       ┆ f64      ┆ str        │
│          ┆ f64         ┆          ┆ f64         ┆ f64        ┆           ┆          ┆            │
╞══════════╪═════════════╪══════════╪═════════════╪════════════╪═══════════╪══════════╪════════════╡
│ 0.533521 ┆ 0.457129    ┆ 0.289741 ┆ 0.654374    ┆ 0.530102   ┆ 0.680077  ┆ 0.524157 ┆ cross-enco │
│          ┆             ┆          ┆             ┆            ┆           ┆          ┆ der/nli-de │
│          ┆             ┆          ┆             ┆            ┆           ┆          ┆ berta-v3-s │
│          ┆             ┆          ┆             ┆            ┆           

In [9]:
all_temp_dfs = []
for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/ksc_synth'):
    if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/ksc_synth/{combination}'):
        combination_splits = combination.split('_')
        dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
        if 'dementia' not in dataset1 and 'dementia' not in dataset2:
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/ksc_synth/{combination}/{combination}_ksc_metrics_measures.csv')
            temp_df = temp_df.groupby('metric').mean().reset_index()
            temp_df['dataset1'] = dataset1
            temp_df['dataset2'] = dataset2
            temp_df['repetitions'] = repetitions
            all_temp_dfs.append(temp_df)

df_ksc_synth = pd.concat(all_temp_dfs)
ksc_synth = make_output_metric_ranking(df_ksc_synth)


shape: (217, 8)
┌──────────┬─────────────┬──────────┬─────────────┬────────────┬───────────┬──────────┬────────────┐
│ Accuracy ┆ Weighted    ┆ Time     ┆ Monotonicit ┆ Separabili ┆ Linearity ┆ Overall  ┆ Algorithm  │
│ ---      ┆ Accuracy    ┆ ---      ┆ y           ┆ ty         ┆ ---       ┆ ---      ┆ ---        │
│ f64      ┆ ---         ┆ f64      ┆ ---         ┆ ---        ┆ f64       ┆ f64      ┆ str        │
│          ┆ f64         ┆          ┆ f64         ┆ f64        ┆           ┆          ┆            │
╞══════════╪═════════════╪══════════╪═════════════╪════════════╪═══════════╪══════════╪════════════╡
│ 0.386115 ┆ 0.338842    ┆ 0.524084 ┆ 0.512762    ┆ 0.366125   ┆ 0.52312   ┆ 0.441842 ┆ typeform/d │
│          ┆             ┆          ┆             ┆            ┆           ┆          ┆ istilbert- │
│          ┆             ┆          ┆             ┆            ┆           ┆          ┆ base-uncas │
│          ┆             ┆          ┆             ┆            ┆           

In [10]:
df_all = pd.concat([df_ksc, df_ksc_synth])

In [11]:
def make_dataset_dependent_results(input_df):
    metric_cols = [
        "Accuracy",
        "Weighted Accuracy",
        "Time",
        "Monotonicity",
        "Separability",
        "Linearity"
    ]
    dataset_grouped_dfs = []
    for dataset in dataset_characteristics_df['Dataset']:
        temp_df = input_df[
            (input_df['dataset1'] == dataset) | 
            (input_df['dataset2'] == dataset)
        ].copy()
        scaler = MinMaxScaler()
        temp_df[metric_cols] = scaler.fit_transform(temp_df[metric_cols])
        # normalize here
        temp_df['overall'] = temp_df[metric_cols].mean(axis=1)
        dataset_grouped_temp = temp_df.groupby('metric')[metric_cols + ['overall']].mean().sort_values(by='overall', ascending=False)
        dataset_grouped_temp['dataset'] = dataset
        dataset_grouped_dfs.append(dataset_grouped_temp)

    grouped_df = pd.concat(dataset_grouped_dfs)
    return grouped_df

In [12]:
ksc_synth

Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity,Overall,Algorithm
f64,f64,f64,f64,f64,f64,f64,str
0.386115,0.338842,0.524084,0.512762,0.366125,0.52312,0.441842,"""typeform/distilbert-base-uncas…"
0.383471,0.339942,0.260729,0.510084,0.380866,0.527557,0.400442,"""typeform/distilbert-base-uncas…"
0.335137,0.304124,0.505187,0.461584,0.330901,0.461428,0.399727,"""typeform/distilbert-base-uncas…"
0.339508,0.299435,0.499538,0.471655,0.311358,0.475858,0.399559,"""typeform/distilbert-base-uncas…"
0.38156,0.336976,0.261945,0.507704,0.368117,0.518252,0.395759,"""typeform/distilbert-base-uncas…"
…,…,…,…,…,…,…,…
0.264723,0.244126,0.107717,0.378027,0.277842,0.384975,0.276235,"""cross-encoder/nli-deberta-v3-s…"
0.270928,0.250964,0.068446,0.392661,0.269446,0.398123,0.275095,"""typeform/distilbert-base-uncas…"
0.274099,0.249486,0.050701,0.39488,0.270062,0.399341,0.273095,"""cross-encoder/nli-deberta-v3-s…"


In [13]:
grouped_ksc = make_dataset_dependent_results(df_ksc)
grouped_ksc_synth = make_dataset_dependent_results(df_ksc_synth)
all_grouped = make_dataset_dependent_results(df_all)

In [14]:
grouped_ksc

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity,overall,dataset
metric,,,,,,,,
cross-encoder/nli-deberta-v3-small__prompt2,0.609026,0.555738,0.383714,0.682264,0.583482,0.693194,0.584570,atis
typeform/distilbert-base-uncased-mnli__prompt5,0.554960,0.485436,0.501378,0.621681,0.496013,0.639171,0.549773,atis
cross-encoder/nli-deberta-v3-small__prompt4,0.564884,0.507450,0.385903,0.635174,0.533665,0.649091,0.546028,atis
cross-encoder/nli-deberta-v3-small__prompt1_prompt2,0.586921,0.520879,0.194286,0.681600,0.590694,0.700150,0.545755,atis
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli__prompt2,0.579697,0.515056,0.217157,0.686680,0.555439,0.700305,0.542389,atis
...,...,...,...,...,...,...,...,...
typeform/distilbert-base-uncased-mnli__prompt1_prompt4,0.345977,0.327343,0.306445,0.386643,0.337868,0.395599,0.349979,yahoo
typeform/distilbert-base-uncased-mnli__prompt1_prompt3,0.348218,0.346335,0.311903,0.370089,0.332071,0.383400,0.348669,yahoo
typeform/distilbert-base-uncased-mnli__prompt1_prompt2_prompt3,0.361513,0.349424,0.205805,0.406226,0.347884,0.403570,0.345737,yahoo


In [15]:
all_grouped.to_excel('./allDatasetResults.xlsx')

In [16]:
for algorithm in ['MAUVE', 'ZERO']:
    for grouped_name, grouped_df in [('grouped_ksc', grouped_ksc), ('grouped_ksc_synth', grouped_ksc_synth), ('all_grouped', all_grouped)]:
        print(algorithm, grouped_name)
        print(pl.from_pandas(grouped_df[grouped_df.index ==algorithm]))
        


MAUVE grouped_ksc
shape: (0, 8)
┌──────────┬──────────┬──────┬──────────────┬──────────────┬───────────┬─────────┬─────────┐
│ Accuracy ┆ Weighted ┆ Time ┆ Monotonicity ┆ Separability ┆ Linearity ┆ overall ┆ dataset │
│ ---      ┆ Accuracy ┆ ---  ┆ ---          ┆ ---          ┆ ---       ┆ ---     ┆ ---     │
│ f64      ┆ ---      ┆ f64  ┆ f64          ┆ f64          ┆ f64       ┆ f64     ┆ str     │
│          ┆ f64      ┆      ┆              ┆              ┆           ┆         ┆         │
╞══════════╪══════════╪══════╪══════════════╪══════════════╪═══════════╪═════════╪═════════╡
└──────────┴──────────┴──────┴──────────────┴──────────────┴───────────┴─────────┴─────────┘
MAUVE grouped_ksc_synth
shape: (0, 8)
┌──────────┬──────────┬──────┬──────────────┬──────────────┬───────────┬─────────┬─────────┐
│ Accuracy ┆ Weighted ┆ Time ┆ Monotonicity ┆ Separability ┆ Linearity ┆ overall ┆ dataset │
│ ---      ┆ Accuracy ┆ ---  ┆ ---          ┆ ---          ┆ ---       ┆ ---     ┆ ---     │


In [17]:
ksc = ksc.to_pandas()
ksc['Test Type'] = 'Real'
ksc_synth = ksc_synth.to_pandas()
ksc_synth['Test Type'] = 'Synth'

combined = pd.concat([ksc, ksc_synth])

In [18]:
pivot = combined.pivot(index="Algorithm", columns="Test Type")
pivot

Accuracy            \
Test Type                                               Real     Synth   
Algorithm                                                                
cross-encoder/nli-deberta-v3-small__prompt1         0.498297  0.305700   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.528271  0.341668   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.522163  0.316328   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.529178  0.324247   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.518476  0.288798   
...                                                      ...       ...   
valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.466297  0.345664   
valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.482229  0.364024   
valhalla/distilbart-mnli-12-3__prompt4              0.427696  0.266095   
valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.470968  0.345088   
valhalla/distilbart-mnli-12-3__prompt5              0.503611  0.370929   

                                                   Weighted Accuracy  \
Test Type                                                       Real   
Algorithm                                                              
cross-encoder/nli-deberta-v3-small__prompt1                 0.416435   
cross-encoder/nli-deberta-v3-small__prompt1_pro...          0.449709   
cross-encoder/nli-deberta-v3-small__prompt1_pro...          0.443110   
cross-encoder/nli-deberta-v3-small__prompt1_pro...          0.451995   
cross-encoder/nli-deberta-v3-small__prompt1_pro...          0.436959   
...                                                              ...   
valhalla/distilbart-mnli-12-3__prompt3_prompt4_...          0.391456   
valhalla/distilbart-mnli-12-3__prompt3_prompt5              0.404522   
valhalla/distilbart-mnli-12-3__prompt4                      0.353166   
valhalla/distilbart-mnli-12-3__prompt4_prompt5              0.394548   
valhalla/distilbart-mnli-12-3__prompt5                      0.422037   

                                                                  Time  \
Test Type                                              Synth      Real   
Algorithm                                                                
cross-encoder/nli-deberta-v3-small__prompt1         0.281842  0.285095   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.310369  0.145838   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.286846  0.098227   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.296145  0.072878   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.258750  0.058012   
...                                                      ...       ...   
valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.317032  0.043933   
valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.329888  0.068167   
valhalla/distilbart-mnli-12-3__prompt4              0.243000  0.133475   
valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.316005  0.066718   
valhalla/distilbart-mnli-12-3__prompt5              0.344953  0.137611   

                                                             Monotonicity  \
Test Type                                              Synth         Real   
Algorithm                                                                   
cross-encoder/nli-deberta-v3-small__prompt1         0.411802     0.621266   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.211909     0.649256   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.143082     0.644959   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.106287     0.647797   
cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.084740     0.642041   
...                                                      ...          ...   
valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.062389     0.584989   
valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.096916     0.599442   
valhalla/distilbart-mnli-12-3__prompt4              0.188131     0.539747   
valhalla/distilbart-mnli-12-3__prompt4_prompt

In [19]:
averages = {}
for column in pivot.columns:
    if column[0] in averages:
        for metric, value in zip(pivot[column].index, pivot[column]):
            averages[column[0]][metric].append(value)
        
    else:
        averages[column[0]] = {}
        for metric, value in zip(pivot[column].index, pivot[column]):
            averages[column[0]][metric] = [value]

final = {}
for metric, values in averages.items():
    final[metric] = {}
    for algorithm, result in values.items():
        final[metric][algorithm] = (sum(result))/(len(result))
        
pd.DataFrame(final).sort_values(by='Overall', ascending=False)

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity,Overall
typeform/distilbert-base-uncased-mnli__prompt5,0.412743,0.352324,0.453620,0.530697,0.387670,0.550337,0.447899
cross-encoder/nli-deberta-v3-small__prompt4,0.428905,0.369590,0.356949,0.539249,0.400577,0.553570,0.441473
cross-encoder/nli-deberta-v3-small_valhalla/distilbart-mnli-12-3__prompt2,0.463531,0.407118,0.109735,0.583597,0.468149,0.599914,0.438674
cross-encoder/nli-deberta-v3-small_valhalla/distilbart-mnli-12-3__prompt1_prompt2,0.469783,0.409659,0.053808,0.590447,0.470343,0.606178,0.433370
cross-encoder/nli-deberta-v3-small__prompt2,0.415224,0.359720,0.355431,0.528827,0.398016,0.542451,0.433278
...,...,...,...,...,...,...,...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt4,0.371303,0.317573,0.087720,0.491584,0.353957,0.499053,0.353532
typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt2_prompt3_prompt4,0.374186,0.323888,0.037495,0.493606,0.378953,0.506501,0.352438
typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt3_prompt4,0.365696,0.316467,0.058597,0.489396,0.363145,0.506525,0.349971
typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt4,0.356124,0.309586,0.117552,0.473770,0.341252,0.485961,0.347374
